In [1]:
!git clone https://github.com/fatimasajid31/urdu-ocr-codesaviours-si26-fatima.git
!ls urdu-ocr-codesaviours-si26-fatima

fatal: destination path 'urdu-ocr-codesaviours-si26-fatima' already exists and is not an empty directory.
 data				       SI26_Week2_Fatima.ipynb
 README.md			       SI26_Week3_Fatima.ipynb
 SI26-Week1-fatima.ipynb	       urdu-ocr-codesaviours-si26-fatima
'SI26_Week1_fatima_ipynb_ (1).ipynb'


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor, base_folder=''):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.base_folder = base_folder
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = f"{self.base_folder}/{row['image']}" if self.base_folder else row['image']
        image = Image.open(image_path).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model_name = 'Hammad712/troce-urdu-model2-v1'

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained(model_name)
model     = VisionEncoderDecoderModel.from_pretrained(model_name)
model     = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model loaded successfully!
Model parameters: 459,852,320


In [4]:
from sklearn.model_selection import train_test_split

csv_path = 'urdu-ocr-codesaviours-si26-fatima/data/labels.csv'
base_folder = 'urdu-ocr-codesaviours-si26-fatima'

df = pd.read_csv(csv_path)
print(f'Total samples: {len(df)}')

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_df.to_csv('train_labels.csv', index=False)
test_df.to_csv('test_labels.csv', index=False)

print(f'Train samples: {len(train_df)}')
print(f'Test samples: {len(test_df)}')

train_dataset = UrduOCRDataset('train_labels.csv', processor, base_folder=base_folder)
test_dataset  = UrduOCRDataset('test_labels.csv', processor, base_folder=base_folder)

Total samples: 224
Train samples: 201
Test samples: 23
Dataset loaded: 201 samples
Dataset loaded: 23 samples


In [5]:
from transformers import AdamW

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1)

optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 201
Ready to train!


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [12]:
import torch
import gc
from jiwer import cer

scaler = torch.cuda.amp.GradScaler()
num_epochs = 8
best_cer = float('inf')

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels       = batch['labels'].to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss    = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        del pixel_values, labels, outputs, loss
        if batch_idx % 20 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} | Train Loss: {avg_loss:.4f}')

    # Evaluate on test set after every epoch
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels']
            generated_ids = model.generate(pixel_values)
            preds.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
            actuals.extend(processor.batch_decode(labels, skip_special_tokens=True))
            del pixel_values, generated_ids
            torch.cuda.empty_cache()

    epoch_cer = cer(actuals, preds)
    print(f'Epoch {epoch + 1} | Test CER: {epoch_cer:.3f}')

    if epoch_cer < best_cer:
        best_cer = epoch_cer
        model.save_pretrained('best_model')
        processor.save_pretrained('best_model')
        print(f'>> New best model saved (CER: {epoch_cer:.3f})')

print(f'\nTraining complete! Best CER achieved: {best_cer:.3f}')

/tmp/ipykernel_25483/2803030627.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_25483/2803030627.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch 1/8
------------------------------
Epoch 1 | Train Loss: 0.0160


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 1}


Epoch 1 | Test CER: 1.002
>> New best model saved (CER: 1.002)

Epoch 2/8
------------------------------


/tmp/ipykernel_25483/2803030627.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

Epoch 4 | Test CER: 0.955
>> New best model saved (CER: 0.955)

Epoch 5/8
------------------------------


/tmp/ipykernel_25483/2803030627.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 5 | Train Loss: 0.0142


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Epoch 5 | Test CER: 1.033

Epoch 6/8
------------------------------
Epoch 6 | Train Loss: 0.0173
Epoch 6 | Test CER: 0.981

Epoch 7/8
------------------------------
Epoch 7 | Train Loss: 0.0217
Epoch 7 | Test CER: 1.047

Epoch 8/8
------------------------------
Epoch 8 | Train Loss: 0.0245
Epoch 8 | Test CER: 1.127

Training complete! Best CER achieved: 0.955


In [13]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model = VisionEncoderDecoderModel.from_pretrained('best_model').to(device)
processor = TrOCRProcessor.from_pretrained('best_model')
print('Best model loaded for final evaluation')

Best model loaded for final evaluation


In [15]:
model.eval()

print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total   = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels       = batch['labels']

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual:    {actual}')
            print()

        del pixel_values, generated_ids
        torch.cuda.empty_cache()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===

Predicted: گامصبر کا پھل میٹھا ہوتا ہے
Actual:    محنت میں عظمت ہے

Predicted: سعلم کے بغیر ترقی ممکن نہیں
Actual:    قوم کی ترقی تعلیم میں ہے

Predicted: گامہمسائیوں کے ساتھ اچھا سلوک کریں
Actual:    کامیاب لوگ ہمیشہ محنتی ہوتے ہیں

Predicted: سپھول خوشبو اور خوبصورتی کی علامت ہیں
Actual:    دوسروں کے کام آنا بڑی نیکی ہے

Predicted: گام3
Actual:    2

Predicted: سہر کام میں برکت نیت پر ہے
Actual:    استاد کا احترام کرنا فرض ہے

Predicted: سعلم حاصل کرنا ہر مسلمان پر فرض ہے
Actual:    سیالکوٹ

Predicted: گامدوستی میں اعتماد بہت ضروری ہے
Actual:    پانی زندگی کے لیے بہت ضروری ہے

Predicted: گام10
Actual:    1

Predicted: گام5
Actual:    4

Predicted:  پرایران میں بے امنی، پاکستانی شہریوں کو غیر ضروری سفر سے گریز کی ہدایت
Actual:    پاکستان میں تیز انٹرنیٹ اور 5جی کیلئے 600 میگا ہرٹز اسپیکٹرم کی نیلامی اگلے ماہ ہوگی

Predicted:  پرصبر کا پھل میٹھا ہوتا ہے
Actual:    اساتذہ کا احترام کرنا چاہیے

Predicted: گام5
Actual:    1

Predicted: سسرخ و عمل ک

In [16]:
from jiwer import cer

model.eval()
all_predictions = []
all_actuals = []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        all_predictions.extend(generated_text)
        all_actuals.extend(actual_text)

        del pixel_values, generated_ids
        torch.cuda.empty_cache()

character_error_rate = cer(all_actuals, all_predictions)
character_accuracy = (1 - character_error_rate) * 100

print(f'Character Error Rate (CER): {character_error_rate:.3f}')
print(f'Character-level Accuracy: {character_accuracy:.1f}%')

Character Error Rate (CER): 0.955
Character-level Accuracy: 4.5%


**EXPAND MY DATASET TO 10,000 bold text BECAUSE I TIRED 6 TIMES AND STILL IT'S GIVING 0 ACCURACY.**

In [18]:
!pip install arabic_reshaper python-bidi --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 14.6 MB/s eta 0:00:00


In [20]:
!wget -q https://github.com/google/fonts/raw/main/ofl/notonastaliqurdu/NotoNastaliqUrdu-Regular.ttf -O nastaliq.ttf
!wget -q https://github.com/google/fonts/raw/main/ofl/notonaskharabic/NotoNaskhArabic-Regular.ttf -O naskh.ttf

import os
print('Nastaliq downloaded:', os.path.exists('nastaliq.ttf'))
print('Naskh downloaded:', os.path.exists('naskh.ttf'))

Nastaliq downloaded: True
Naskh downloaded: True


In [23]:
!ls -la nastaliq.ttf naskh.ttf
!file nastaliq.ttf naskh.ttf

-rw-r--r-- 1 root root 0 Jul 23 10:07 naskh.ttf
-rw-r--r-- 1 root root 0 Jul 23 10:07 nastaliq.ttf
nastaliq.ttf: empty
naskh.ttf:    empty


In [24]:
!apt-get install -y fonts-noto-core fonts-noto-extra -q
!find /usr/share/fonts -iname "*nastaliq*"
!find /usr/share/fonts -iname "*naskh*"

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  fonts-noto-core fonts-noto-extra
0 upgraded, 2 newly installed, 0 to remove and 53 not upgraded.
Need to get 84.6 MB of archives.
After this operation, 386 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-noto-core all 20201225-1build1 [12.2 MB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-noto-extra all 20201225-1build1 [72.4 MB]
Fetched 84.6 MB in 1s (58.6 MB/s)
Selecting previously unselected package fonts-noto-core.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../fonts-noto-core_20201225-1build1_all.deb ...
Unpacking fonts-noto-core (20201225-1build1) ...
Selecting previously unselected package fonts-noto-extra.
Preparing to unpack .../fonts-noto-extra_20201225-1build1_all.deb ...
Unpacking fonts-noto-extra (20201225-1build1) ...
S

In [26]:
!apt-get install -y fonts-noto-core fonts-noto-extra -q
!find /usr/share/fonts -iname "*nastaliq*"
!find /usr/share/fonts -iname "*naskh*"

Reading package lists...
Building dependency tree...
Reading state information...
fonts-noto-core is already the newest version (20201225-1build1).
fonts-noto-extra is already the newest version (20201225-1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
/usr/share/fonts/truetype/noto/NotoNastaliqUrdu-Regular.ttf
/usr/share/fonts/truetype/noto/NotoNastaliqUrdu-Bold.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabic-Bold.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabic-SemiBold.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabic-Medium.ttf


In [28]:
!apt-get install -y fonts-noto-core fonts-noto-extra -q
!find /usr/share/fonts -iname "*nastaliq*"
!find /usr/share/fonts -iname "*naskh*"

Reading package lists...
Building dependency tree...
Reading state information...
fonts-noto-core is already the newest version (20201225-1build1).
fonts-noto-extra is already the newest version (20201225-1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
/usr/share/fonts/truetype/noto/NotoNastaliqUrdu-Regular.ttf
/usr/share/fonts/truetype/noto/NotoNastaliqUrdu-Bold.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabic-Bold.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabic-SemiBold.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabic-Medium.ttf


In [29]:
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display
import pandas as pd
import os
import random

subjects = ["علم", "محنت", "دوستی", "صحت", "وقت", "کتاب", "استاد", "طالب علم",
            "کسان", "ڈاکٹر", "انجینئر", "سچائی", "ایمانداری", "صبر", "امن",
            "محبت", "احترام", "نظم", "اتحاد", "ترقی", "خوشی", "غم", "ہمت", "یقین",
            "قوم", "ملک", "خاندان", "بچے", "بزرگ", "معاشرہ", "قانون", "انصاف",
            "آزادی", "جمہوریت", "تعلیم", "ثقافت", "زبان", "تاریخ", "جغرافیہ", "سائنس",
            "پانی", "ہوا", "زمین", "آسمان", "سورج", "چاند", "ستارے", "دریا",
            "پہاڑ", "درخت", "پھول", "پرندے", "جانور", "موسم", "بارش", "برف",
            "شہر", "گاؤں", "بازار", "گھر", "سکول", "ہسپتال", "مسجد", "دفتر"]

predicates = ["بہت اہم ہے", "زندگی کا حصہ ہے", "کامیابی کی کنجی ہے",
              "ہر انسان کے لیے ضروری ہے", "معاشرے کی بنیاد ہے",
              "قوم کی ترقی میں مدد دیتا ہے", "انسان کو آگے بڑھاتا ہے",
              "دل کو سکون دیتا ہے", "زندگی کو خوبصورت بناتا ہے",
              "مستقبل سنوارتا ہے", "ہمیں بہتر بناتا ہے", "دنیا کو بدل سکتا ہے",
              "ہر دور میں اہمیت رکھتا ہے", "نسلوں کو جوڑتا ہے",
              "امن قائم کرنے میں مدد دیتا ہے", "خوشحالی لاتا ہے",
              "قدرت کا انمول تحفہ ہے", "ہر جگہ نظر آتا ہے", "بہت خوبصورت ہوتا ہے",
              "لوگوں کو متاثر کرتا ہے", "روزمرہ زندگی میں کام آتا ہے"]

connectors = ["اور", "لیکن", "کیونکہ", "اس لیے", ""]

new_sentences = set()
while len(new_sentences) < 10000:
    s1 = random.choice(subjects)
    p1 = random.choice(predicates)
    sentence = f"{s1} {p1}"

    if random.random() > 0.5:
        s2 = random.choice(subjects)
        p2 = random.choice(predicates)
        conn = random.choice(connectors)
        sentence += f" {conn} {s2} {p2}".replace("  ", " ")

    new_sentences.add(sentence)

new_sentences = list(new_sentences)
print(f'Generated {len(new_sentences)} unique sentences')

fonts = [
    '/usr/share/fonts/truetype/noto/NotoNastaliqUrdu-Regular.ttf',
    '/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf'
]
font_sizes = [26, 30, 34, 38, 42]
backgrounds = ['white', '#f5f5dc', '#fffbe6', '#f0f0f0']

output_folder = 'urdu-ocr-codesaviours-si26-fatima/data/synthetic_v2'
os.makedirs(output_folder, exist_ok=True)

new_rows = []
for i, text in enumerate(new_sentences):
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)

    font_file = random.choice(fonts)
    font_size = random.choice(font_sizes)
    bg_color = random.choice(backgrounds)

    font = ImageFont.truetype(font_file, font_size)
    img = Image.new('RGB', (600, 100), color=bg_color)
    draw = ImageDraw.Draw(img)
    draw.text((10, 25), bidi_text, fill='black', font=font)

    filename = f'urdu_v2_{i+1}.png'
    img.save(f'{output_folder}/{filename}')

    new_rows.append({'image': f'data/synthetic_v2/{filename}', 'text': text})

    if (i+1) % 1000 == 0:
        print(f'Progress: {i+1}/{len(new_sentences)} images generated')

print(f'Saved {len(new_rows)} new images to {output_folder}')

existing_csv = 'urdu-ocr-codesaviours-si26-fatima/data/labels.csv'
existing_df = pd.read_csv(existing_csv)
new_df = pd.DataFrame(new_rows)
combined_df = pd.concat([existing_df, new_df], ignore_index=True)
combined_df.to_csv(existing_csv, index=False)

print(f'Total dataset size now: {len(combined_df)} images (was {len(existing_df)})')

Generated 10000 unique sentences
Progress: 1000/10000 images generated
Progress: 2000/10000 images generated
Progress: 3000/10000 images generated
Progress: 4000/10000 images generated
Progress: 5000/10000 images generated
Progress: 6000/10000 images generated
Progress: 7000/10000 images generated
Progress: 8000/10000 images generated
Progress: 9000/10000 images generated
Progress: 10000/10000 images generated
Saved 10000 new images to urdu-ocr-codesaviours-si26-fatima/data/synthetic_v2
Total dataset size now: 10224 images (was 224)


In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split

csv_path = 'urdu-ocr-codesaviours-si26-fatima/data/labels.csv'
base_folder = 'urdu-ocr-codesaviours-si26-fatima'

full_df = pd.read_csv(csv_path)
print(f'Full dataset size: {len(full_df)}')

# Sample 1500 images for tonight's training (keeps full 10k+ in repo for later)
subset_df = full_df.sample(n=1500, random_state=42).reset_index(drop=True)

train_df, test_df = train_test_split(subset_df, test_size=0.1, random_state=42)
train_df.to_csv('train_labels.csv', index=False)
test_df.to_csv('test_labels.csv', index=False)

print(f'Train samples: {len(train_df)}')
print(f'Test samples: {len(test_df)}')

train_dataset = UrduOCRDataset('train_labels.csv', processor, base_folder=base_folder)
test_dataset  = UrduOCRDataset('test_labels.csv', processor, base_folder=base_folder)

Full dataset size: 10224
Train samples: 1350
Test samples: 150
Dataset loaded: 1350 samples
Dataset loaded: 150 samples


In [31]:
from torch.utils.data import DataLoader
from transformers import AdamW

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1)

optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 1350
Ready to train!


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [32]:
import torch
import gc
from jiwer import cer

scaler = torch.cuda.amp.GradScaler()
num_epochs = 5
best_cer = float('inf')

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels       = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss    = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        del pixel_values, labels, outputs, loss
        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {total_loss / (batch_idx + 1):.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} | Train Loss: {avg_loss:.4f}')

    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels']
            generated_ids = model.generate(pixel_values)
            preds.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
            actuals.extend(processor.batch_decode(labels, skip_special_tokens=True))
            del pixel_values, generated_ids
            torch.cuda.empty_cache()

    epoch_cer = cer(actuals, preds)
    print(f'Epoch {epoch + 1} | Test CER: {epoch_cer:.3f}')

    if epoch_cer < best_cer:
        best_cer = epoch_cer
        model.save_pretrained('best_model')
        processor.save_pretrained('best_model')
        print(f'>> New best model saved (CER: {epoch_cer:.3f})')

print(f'\nTraining complete! Best CER achieved: {best_cer:.3f}')

/tmp/ipykernel_25483/3235839313.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_25483/3235839313.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch 1/5
------------------------------
  Batch 0/1350 | Loss: 0.6671
  Batch 100/1350 | Loss: 0.3489
  Batch 200/1350 | Loss: 0.2755
  Batch 300/1350 | Loss: 0.2401
  Batch 400/1350 | Loss: 0.2192
  Batch 500/1350 | Loss: 0.2063
  Batch 600/1350 | Loss: 0.1958
  Batch 700/1350 | Loss: 0.1883
  Batch 800/1350 | Loss: 0.1823
  Batch 900/1350 | Loss: 0.1775
  Batch 1000/1350 | Loss: 0.1738
  Batch 1100/1350 | Loss: 0.1708
  Batch 1200/1350 | Loss: 0.1685
  Batch 1300/1350 | Loss: 0.1661
Epoch 1 | Train Loss: 0.1651


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 1}


Epoch 1 | Test CER: 0.809
>> New best model saved (CER: 0.809)

Epoch 2/5
------------------------------


/tmp/ipykernel_25483/3235839313.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Batch 0/1350 | Loss: 0.1462
  Batch 100/1350 | Loss: 0.1278
  Batch 200/1350 | Loss: 0.1312
  Batch 300/1350 | Loss: 0.1364
  Batch 400/1350 | Loss: 0.1359
  Batch 500/1350 | Loss: 0.1366
  Batch 600/1350 | Loss: 0.1352
  Batch 700/1350 | Loss: 0.1340
  Batch 800/1350 | Loss: 0.1348
  Batch 900/1350 | Loss: 0.1341
  Batch 1000/1350 | Loss: 0.1336
  Batch 1100/1350 | Loss: 0.1343
  Batch 1200/1350 | Loss: 0.1340
  Batch 1300/1350 | Loss: 0.1342
Epoch 2 | Train Loss: 0.1344


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Epoch 2 | Test CER: 0.882

Epoch 3/5
------------------------------
  Batch 0/1350 | Loss: 0.0885
  Batch 100/1350 | Loss: 0.1301
  Batch 200/1350 | Loss: 0.1319
  Batch 300/1350 | Loss: 0.1310
  Batch 400/1350 | Loss: 0.1310
  Batch 500/1350 | Loss: 0.1311
  Batch 600/1350 | Loss: 0.1300
  Batch 700/1350 | Loss: 0.1300
  Batch 800/1350 | Loss: 0.1293
  Batch 900/1350 | Loss: 0.1295
  Batch 1000/1350 | Loss: 0.1299
  Batch 1100/1350 | Loss: 0.1296
  Batch 1200/1350 | Loss: 0.1307
  Batch 1300/1350 | Loss: 0.1314
Epoch 3 | Train Loss: 0.1313
Epoch 3 | Test CER: 0.938

Epoch 4/5
------------------------------
  Batch 0/1350 | Loss: 0.1245
  Batch 100/1350 | Loss: 0.1334
  Batch 200/1350 | Loss: 0.1350
  Batch 300/1350 | Loss: 0.1340
  Batch 400/1350 | Loss: 0.1362
  Batch 500/1350 | Loss: 0.1352
  Batch 600/1350 | Loss: 0.1341
  Batch 700/1350 | Loss: 0.1335
  Batch 800/1350 | Loss: 0.1324
  Batch 900/1350 | Loss: 0.1330
  Batch 1000/1350 | Loss: 0.1326
  Batch 1100/1350 | Loss: 0.1327
 

In [33]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model = VisionEncoderDecoderModel.from_pretrained('best_model').to(device)
processor = TrOCRProcessor.from_pretrained('best_model')
print('Best model loaded for final evaluation')

Best model loaded for final evaluation


In [38]:
model.eval()

print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total   = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels       = batch['labels']

        generated_ids = model.generate(pixel_values, max_length=32)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual:    {actual}')
            print()

        del pixel_values, generated_ids
        torch.cuda.empty_cache()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===

Predicted: پرندے قدرت کا انمول تحفہ ہے کیونکہ زمین ہر انسان کے لیے ضروری ہیں
Actual:    پرندے قدرت کا انمول تحفہ ہے

Predicted: پرندے ہر انسان کے لیے ضروری ہے
Actual:    ہسپتال ہر انسان کے لیے ضروری ہے

Predicted:  پردوستی قدرت کا انمول تحفہ ہے کیونکہ زمین ہر انسان کے لیے ضروری ہیں
Actual:    پھول مستقبل سنوارتا ہے لیکن ملک انسان کو آگے بڑھاتا ہے

Predicted:  پرشہر ہر انسان کے لیے ضروری ہے کیونکہ زمین قدرت کا انمول تحفہ سے جوڑتا تھا صحت دنیا کو بدل سکتا کاری رکھتا بن علم کامیابی کی ترقی میں
Actual:    پرندے ہر انسان کے لیے ضروری ہے خاندان ہمیں بہتر بناتا ہے

Predicted: پرندے قدرت کا انمول تحفہ ہے کیونکہ زمین ہر انسان کے لیے ضروری ہیں
Actual:    انجینئر مستقبل سنوارتا ہے کیونکہ صبر دل کو سکون دیتا ہے

Predicted: پرندے قدرت کا انمول تحفہ ہے کیونکہ زمین ہر انسان کے لیے ضروری ہیں
Actual:    درخت روزمرہ زندگی میں کام آتا ہے اور احترام قوم کی ترقی میں مدد دیتا ہے

Predicted:  پردوستی ہر انسان کے لیے ضروری ہے کیونکہ زمین قدرت کا انمول تحفہ سے جوڑتا تھا

bold text **STILL GIVING SAME ACCURACY 0 PERCENT EVEN AFTER AFTER MY DATASET INTO 10.000**

In [39]:
from jiwer import cer

model.eval()
all_predictions = []
all_actuals = []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        generated_ids = model.generate(pixel_values, max_length=32)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        all_predictions.extend(generated_text)
        all_actuals.extend(actual_text)

        del pixel_values, generated_ids
        torch.cuda.empty_cache()

character_error_rate = cer(all_actuals, all_predictions)
character_accuracy = (1 - character_error_rate) * 100

print(f'Character Error Rate (CER): {character_error_rate:.3f}')
print(f'Character-level Accuracy: {character_accuracy:.1f}%')

Character Error Rate (CER): 0.898
Character-level Accuracy: 10.2%


In [40]:
# Sanity check: run just 1 test image
model.eval()
sample_batch = next(iter(test_loader))
pixel_values = sample_batch['pixel_values'].to(device)
labels = sample_batch['labels']

with torch.no_grad():
    generated_ids = model.generate(pixel_values)

pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
actual = processor.batch_decode(labels, skip_special_tokens=True)[0]

print(f'Predicted: {pred}')
print(f'Actual:    {actual}')

Predicted: پرندے قدرت کا انمول تحفہ ہے کیونکہ زمین ہر انسان کے لیے ضروری ہیں
Actual:    پرندے قدرت کا انمول تحفہ ہے
